# 将微带线模型与测量结果关联## 目标本示例的目的是将微带线模型与在 1MHz 到 5GHz 范围内四个频率数量级的测量结果关联起来。## 计划1. 测量两种不同长度的微带线；2. 使用多线法计算介电材料的频率相关的相对介电常数和损耗角；3. 通过优化将微带线模型拟合到计算出的参数；4. 通过嵌入连接器并将其与测量结果进行比较，来验证结果。

In [ ]:
%load_ext autoreload
%autoreload 2
import matplotlib.pyplot as plt
import numpy as np
from numpy import absolute, log10, real, sum
from scipy.optimize import minimize

import skrf as rf

rf.stylely()

## 测量两条不同长度的微带线测量于 2017 年 3 月 21 日在安立思 MS46524B 20GHz VNA 上进行。设置是 1MHz 到 10GHz 的线性频率扫描，共 10,000 个点。输出功率为 0dBm，中频带宽为 1kHz，未使用平均或平滑处理。感兴趣的频率范围限制在 1MHz 到 5GHz，但测量频率高达 10GHz。MSLxxx 是一种 L 长的、W 宽的、T 厚的铜微带线，位于 H 高的基板上，基板下方是接地层。| 名称 | L (mm) | W (mm) | H (mm) | T (um) | 基板 || :--- | ---: | ---: | ---: | ---: | :--- || MSL100 | 100 | 3.00 | 1.55 | 50 | FR-4 || MSL200 | 200 | 3.00 | 1.55 | 50 | FR-4 |通过机械方式进行图案铣削，侧壁倾斜角度为 45°。提供一个小型的顶部接地层块，通过通孔阵列连接到底部接地层，用于焊接连接器顶部接地引脚，并提供类似于共面波导的从同轴到微带线的过渡。为了设计目的，假设介电常数的相对介电常数约为 4.5。![MSL100 和 MSL200 的示意图，两者都是微带线，MSL200 的长度是 MSL100 的两倍](MSL_CPWG_100_200.jpg “MSL100 和 MSL200”)

In [ ]:
# Load raw measurements
MSL100_raw = rf.Network('MSL100.s2p')
MSL200_raw = rf.Network('MSL200.s2p')

# Keep only the data from 1MHz to 5GHz
MSL100 = MSL100_raw['1-5000mhz']
MSL200 = MSL200_raw['1-5000mhz']

plt.figure()
plt.title('Measured data')
MSL100.plot_s_db()
MSL200.plot_s_db()
plt.show()

测量数据表明，MSL200的电长度大约是MSL100的两倍。与MSL100相比，MSL200的回波损耗峰值之间的频率间隔大约是其一半。如果忽略较小的连接器长度，这与物理尺寸是一致的。MSL200的插入损耗也大约是MSL100的两倍，这与较长的路径导致更大的衰减是一致的。通常认为，对于微带线，-20dB以下的回波损耗是可接受的，这对应于1%的功率被反射。

## 通过多线法提取介电材料的有效相对介电常数从测量的传输参数的相位中减去一个值。由于两个DUT（器件待测）上都存在连接器，因此连接器的长度影响会被抵消，剩余的相位差与两个DUT长度的差异相关。已知物理长度 $\Delta L$ 和相位 $\Delta \phi$，可以根据以下关系计算出有效相对介电常数 $\epsilon_{r,eff}$：$$\left\{ \begin{array}{ll}\lambda = \frac{c_0}{f \cdot \sqrt{\epsilon_{r,eff}}} \\\phi = \frac{2\pi L}{\lambda}\end{array} \right. \implies\epsilon_{r,eff} = \left( \frac{\Delta \phi \cdot c_0}{2 \pi f \cdot \Delta L} \right)^2 $$基于相同的原理，两个DUT的插入损耗之差可以得到长度差异的插入损耗，从而抵消连接器的影响。

In [ ]:
c0       = 3e8
f        = MSL100.f
deltaL   = 0.1
deltaPhi = np.unwrap(np.angle(MSL100.s[:,1,0])) - np.unwrap(np.angle(MSL200.s[:,1,0]))
Er_eff   = np.power(deltaPhi * c0 / (2 * np.pi * f * deltaL), 2)
Loss_mea = 20 * log10(absolute(MSL200.s[:,1,0] / MSL100.s[:,1,0]))

plt.figure()
plt.suptitle('Effective relative permittivity and loss')
plt.subplot(2,1,1)
plt.plot(f * 1e-9, Er_eff)
plt.ylabel(r'$\epsilon_{r,eff}$')

plt.subplot(2,1,2)
plt.plot(f * 1e-9, Loss_mea)
plt.xlabel('Frequency (GHz)')
plt.ylabel('Insertion Loss (dB)')
plt.show()

该几何结构的有效相对介电常数在低频时表现出色散效应，可以通过宽带德拜模型（例如 skrf 中 *Djordjevic/Svensson* 实现的微带线介质）进行建模。然后，该值随频率缓慢增加，大致对应于 *Kirschning 和 Jansen* 色散模型。插入损耗似乎与频率成正比，这表明介电损耗占主导地位。导体损耗与频率的平方根相关。忽略辐射损耗。

## 通过优化，将微带线模型拟合到计算出的参数### 有效相对介电常数将微带线介质模型与所测量微带线的物理尺寸相结合，通过优化基板在 1 GHz 时的 $\epsilon_r$ 和 tanδ，将其拟合到计算出的 $\epsilon_{r,eff}$。用于考虑参数随频率变化的色散模型为 *Djordjevic/Svensson* 和 *Kirschning and Jansen*。

In [ ]:
from skrf.media import MLine

W   = 3.00e-3
H   = 1.55e-3
T   = 50e-6
L   = 0.1
Er0   = 4.5
tand0 = 0.02
f_epr_tand = 1e9
x0 = [Er0, tand0]

def model(x, freq, Er_eff, L, W, H, T, f_epr_tand, Loss_mea):
    ep_r = x[0]
    tand = x[1]
    m = MLine(frequency=freq, z0_port=50, w=W, h=H, t=T,
        ep_r=ep_r, mu_r=1, rho=1.712e-8, tand=tand, rough=0.15e-6,
        f_low=1e3, f_high=1e12, f_epr_tand=f_epr_tand,
        diel='djordjevicsvensson', disp='kirschningjansen')
    DUT  = m.line(L, 'm')
    Loss_mod = 20 * log10(absolute(DUT.s[:,1,0]))
    return sum((real(m.ep_reff_f) - Er_eff)**2) + 0.01*sum((Loss_mod - Loss_mea)**2)

res = minimize(model, x0, args=(MSL100.frequency, Er_eff, L, W, H, T, f_epr_tand, Loss_mea),
               bounds=[(4.2, 4.7), (0.001, 0.1)])
Er   = res.x[0]
tand = res.x[1]

print(f'Er={Er:.3f}, tand={tand:.4f} at {f_epr_tand * 1e-9:.1f} GHz.')

为了验证模型的有效性，将模型数据与计算出的参数进行比较。

In [ ]:
m = MLine(frequency=MSL100.frequency, z0_port=50, w=W, h=H, t=T,
        ep_r=Er, mu_r=1, rho=1.712e-8, tand=tand, rough=0.15e-6,
        f_low=1e3, f_high=1e12, f_epr_tand=f_epr_tand,
        diel='djordjevicsvensson', disp='kirschningjansen')
DUT  = m.line(L, 'm')
DUT.name = 'DUT'
Loss_mod = 20 * log10(absolute(DUT.s[:,1,0]))

plt.figure()
plt.suptitle('Measurement vs Model')
plt.subplot(2,1,1)
plt.plot(f * 1e-9, Er_eff, label='Measured')
plt.plot(f * 1e-9, real(m.ep_reff_f), label='Model')
plt.ylabel(r'$\epsilon_{r,eff}$')
plt.legend()

plt.subplot(2,1,2)
plt.plot(f * 1e-9, Loss_mea, label='Measured')
plt.plot(f * 1e-9, Loss_mod, label='Model')
plt.xlabel('Frequency (GHz)')
plt.ylabel('Insertion Loss (dB)')
plt.legend()
plt.show()

模型结果表明，它与测得的 $\epsilon_{r,eff}$ 和插入损耗值之间存在合理的对应关系。

## 检查结果如果将模型绘制在与相同长度的测量结果的图上，图表将显示两者之间没有一致性。这是因为模型没有考虑到连接器的影响。

In [ ]:
plt.figure()
plt.title('Measured vs modelled data')
MSL100.plot_s_db()
DUT.plot_s_db(0, 0, color='k')
DUT.plot_s_db(1, 0, color='k')
plt.show()


### 连接器延迟和损耗估算通过将连接器的相位贡献与频率进行拟合，可以估算连接器的延迟。通过从相同长度的测量结果中减去未包含连接器时的相位和损耗，可以计算出两个连接器的相位和损耗。

In [ ]:
phi_conn = np.unwrap(np.angle(MSL100.s[:,1,0])) + deltaPhi
z = np.polyfit(f, phi_conn, 1)
p = np.poly1d(z)
delay = -z[0]/(2*np.pi)/2
print(f'Connector delay: {delay * 1e12:.0f} ps')

loss_conn_db = 20 * log10(absolute(MSL100.s[:,1,0])) - Loss_mea
alpha = 1.6*np.log(10)/20 * np.sqrt(f/1e9)
beta  = 2*np.pi*f/c0
gamma = alpha + 1j*beta
mf = rf.media.DefinedGammaZ0(m.frequency, z0_port=50, z0=53.5, gamma=gamma)
left = mf.line(delay*1e9, 'ns')
right = left.flipped()
check = left ** right

plt.figure()
plt.suptitle('Connector effects')
plt.subplot(2,1,1)
plt.plot(f * 1e-9, phi_conn, label='measured')
plt.plot(f * 1e-9, np.unwrap(np.angle(check.s[:,1,0])), label='model')
plt.ylabel('phase (rad)')
plt.legend()

plt.subplot(2,1,2)
plt.plot(f * 1e-9, loss_conn_db, label='Measured')
plt.plot(f * 1e-9, 20*np.log10(np.absolute(check.s[:,1,0])), label='model')
plt.xlabel('Frequency (GHz)')
plt.ylabel('Insertion Loss (dB)')
plt.legend()
plt.show()

模型的相位表现出良好的吻合度，而插入损耗似乎也具有合理的数值，并且相对较小。

### 通过时域反射测量法调整连接器阻抗使用测量和模型的时域阶跃响应来调整连接器模型的特性阻抗。图表显示，连接器具有电感特性（正峰值），而微带线则略微具有过多的电容特性（负平台）。通过反复试验的方式调整连接器的特性阻抗，直到达到合理的匹配。也可以使用优化方法。

In [ ]:
mod = left ** DUT ** right
mod.name = 'Model'

MSL100_dc = MSL100.extrapolate_to_dc(kind='cubic')
DUT_dc = mod.extrapolate_to_dc(kind='cubic')

plt.figure()
plt.suptitle('Left-right and right-left TDR')
plt.subplot(2,1,1)
MSL100_dc.plot_z_time_step(0, 0)
DUT_dc.plot_z_time_step(0, 0)
plt.xlim(-2, 4)

plt.subplot(2,1,2)
MSL100_dc.plot_z_time_step(1, 1)
DUT_dc.plot_z_time_step(1, 1)
plt.xlim(-2, 4)

plt.tight_layout()
plt.show()

### 最终比较

In [ ]:
plt.figure()
plt.title('Measured vs modelled data')
MSL100.plot_s_db()
mod.name = 'Model'
mod.plot_s_db(0, 0, color='k')
mod.plot_s_db(1, 0, color='k')

plt.show()

该图显示模型与测量数据之间存在相当好的一致性。该模型很好地表示了在 1 MHz 到 5 GHz 之间的 DUT。在较高频率下，模型开始偏离测量值。该模型未能捕捉到辐射损耗或复杂的铜表面粗糙度等效应。较小的几何结构，例如顶层接地平面块，也可能开始产生影响，因为随着频率的增加，它们在电学上变得更长。作为比较，在空气中，5 GHz 波长的波长为 60 毫米，MSL100 线路的长度为 100 毫米。DUT 本身在高于某些 GHz 的频率下，在电学上也是较长的。